# Latency, Deadlines, and Deadline-Miss Diagnostics Lab


## Purpose

This lab evaluates end-to-end latency, deadline misses, and latency margins for real-time AI tasks.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 1500

tasks = pd.DataFrame({
    "task_id": [f"rtai-{i:04d}" for i in range(1, n + 1)],
    "task_type": rng.choice(
        ["perception", "tracking", "planning", "control", "safety_monitor"],
        size=n,
        p=[0.32, 0.20, 0.18, 0.20, 0.10],
    ),
    "deadline_ms": rng.choice([20, 40, 80, 120], size=n, p=[0.20, 0.35, 0.30, 0.15]),
    "risk_level": rng.choice(["low", "medium", "high"], size=n, p=[0.45, 0.35, 0.20]),
})

latency_profiles = {
    "perception": (18, 8),
    "tracking": (10, 4),
    "planning": (35, 15),
    "control": (8, 3),
    "safety_monitor": (12, 5),
}

sense_latency = rng.normal(loc=5, scale=1.5, size=n)
pre_latency = rng.normal(loc=6, scale=2.0, size=n)

infer_latency = np.array([
    rng.normal(
        loc=latency_profiles[task_type][0],
        scale=latency_profiles[task_type][1],
    )
    for task_type in tasks["task_type"]
])

act_latency = rng.normal(loc=4, scale=1.0, size=n)

tasks["latency_ms"] = np.maximum(
    1,
    sense_latency + pre_latency + infer_latency + act_latency,
)

tasks["deadline_miss"] = tasks["latency_ms"] > tasks["deadline_ms"]
tasks["latency_margin_ms"] = tasks["deadline_ms"] - tasks["latency_ms"]

tasks.head()

In [ ]:
summary = (
    tasks
    .groupby("task_type")
    .agg(
        tasks=("task_id", "count"),
        mean_latency_ms=("latency_ms", "mean"),
        p95_latency_ms=("latency_ms", lambda x: np.percentile(x, 95)),
        p99_latency_ms=("latency_ms", lambda x: np.percentile(x, 99)),
        deadline_miss_rate=("deadline_miss", "mean"),
        mean_latency_margin_ms=("latency_margin_ms", "mean"),
    )
    .reset_index()
)

summary.sort_values("deadline_miss_rate", ascending=False)

## Interpretation

Real-time AI should be evaluated with p95, p99, deadline misses, and latency margins, not average inference time alone.